# GPU Cloud Benchmark — Analysis Notebook

Interactive exploration of benchmark results.  
Run this after a benchmark completes and results are in `../results/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)

RESULTS_DIR = '../results'
COST_RATES_PATH = '../config/gpu_cost_rates.yaml'

## 1. Load and Aggregate Results

In [ ]:
from src.analysis.preprocessor import load_summary_csvs, compute_aggregate_stats

raw_df = load_summary_csvs(RESULTS_DIR)
print(f'Loaded {len(raw_df)} raw runs')
raw_df.head()

In [ ]:
agg_df = compute_aggregate_stats(raw_df)
print(f'{len(agg_df)} aggregated groups')
agg_df

## 2. Cost Analysis

In [ ]:
from src.cost.calculator import load_gpu_rates, compute_cost_metrics

gpu_rates = load_gpu_rates(COST_RATES_PATH)
cost_df = compute_cost_metrics(agg_df, gpu_rates)
cost_df[['gpu_type', 'workload', 'batch_size', 'mean_throughput',
          'cost_per_hour', 'throughput_per_dollar', 'cost_per_1k_samples',
          'cost_efficiency_rank']]

## 3. Throughput Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
chart_df = cost_df.copy()
chart_df['label'] = chart_df['workload'] + ' (bs=' + chart_df['batch_size'].astype(str) + ')'
pivot = chart_df.pivot_table(index='gpu_type', columns='label', values='mean_throughput')
pivot.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='white')
ax.set_ylabel('Throughput (samples/sec)')
ax.set_title('Throughput by GPU and Workload')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Cost Efficiency

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sorted_df = cost_df.sort_values('throughput_per_dollar', ascending=True)
sorted_df['label'] = sorted_df['gpu_type'] + ' / ' + sorted_df['workload']
colors = sns.color_palette('viridis', len(sorted_df))
ax.barh(sorted_df['label'], sorted_df['throughput_per_dollar'], color=colors)
ax.set_xlabel('Throughput per Dollar (samples/$)')
ax.set_title('Cost Efficiency Ranking')
plt.tight_layout()
plt.show()

## 5. Batch Size Scaling

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for (gpu, wl), group in cost_df.groupby(['gpu_type', 'workload']):
    g = group.sort_values('batch_size')
    ax.plot(g['batch_size'], g['mean_throughput'], marker='o', label=f'{gpu} / {wl}', linewidth=2)
ax.set_xlabel('Batch Size')
ax.set_ylabel('Throughput (samples/sec)')
ax.set_title('Throughput Scaling with Batch Size')
ax.set_xscale('log', base=2)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Reproducibility Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
hm_df = cost_df.copy()
hm_df['config'] = hm_df['workload'] + ' bs=' + hm_df['batch_size'].astype(str)
pivot = hm_df.pivot_table(index='gpu_type', columns='config', values='cv_throughput')
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Coefficient of Variation (Throughput)')
plt.tight_layout()
plt.show()

## 7. Environment & Manifest

In [ ]:
from pathlib import Path

manifest_path = Path(RESULTS_DIR) / 'run_manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print('GPU:', manifest.get('gpu_type'))
    print('Total runs:', manifest.get('total_runs'))
    print('Failed:', manifest.get('failed_runs'))
    env = manifest.get('environment', {})
    for k in ['torch_version', 'cuda_version', 'gpu_name', 'nvidia_driver_version']:
        print(f'  {k}: {env.get(k, "N/A")}')
else:
    print('No run_manifest.json found — run the benchmark first.')